# Introducing Cats

Common abstractions for functional programming in **Scala**

## What is Cats?

![cats](https://typelevel.org/cats/img/cats2.png)

 - Library providing *abstractions* for functional programming in Scala
 - Foundation for an ecosystem of *pure*, *typeful* libraries
 - Why **cats**?
     - Concepts coming from *“category theory”*

## What Cats offers

 - range of type classes
 - set of extension methods 
 - general FP primitives
 - (among many other things)

**Purpose:** allow to write general and *extensible* FP code


## How to include Cats in a project?

Including in an `build.sbt` project:

```
ThisBuild / version := "1.1"
ThisBuild / scalaVersion := "3.3.7"
ThisBuild / organization := "ch.hevs"
  libraryDependencies ++= Seq(
    "org.typelevel" %% "cats-core" % "2.13.0" ,
    "org.scalatest" %% "scalatest" % "3.2.14" % Test
)
```

## Importing Cats in this notebook

In [10]:
import $ivy.`org.typelevel::cats-core:2.13.0`

Downloaded https://repo1.maven.org/maven2/org/typelevel/cats-core_3/2.13.0/cats-core_3-2.13.0.pom
Downloaded https://repo1.maven.org/maven2/org/typelevel/cats-kernel_3/2.13.0/cats-kernel_3-2.13.0.pom


import $ivy.$                                


## Using Type Classes

Cats relies heavily on Type Classes (remember?)

 - enable ad-hoc polymorphism, i.e., overloading
 - alternative to subtyping
 - parametric polymorphism + ad-hoc polymorphism.



Example: remeber `Taxable[A]`:

In [1]:
trait Taxable[A]:
  extension(a:A)
    def computeTax:Double

defined trait Taxable

In [2]:
case class Architect(name: String, income:Double)

defined class Architect

In [3]:
given Taxable[Architect] with
  extension(a:Architect) def computeTax = a.income * 0.3

defined object 

## Show

The `toString` method has several problems, e.g. can be called on anything, and do unexpected things:

In [4]:
34.5.toString

res4: String = "34.5"

In [5]:
(new Architect("jim",34)).toString

res5: String = "Architect(jim,34.0)"

In [7]:
(new {val a=34}).toString

res7: String = "ammonite.$sess.cmd7$$anon$1@57a94353"

**Show** provides an alternative to `toString`, only to apply to desired classes.

The signature of the `show` function is given as:
```
def show[A](f: A => String): Show[A]
```

We can use `Show` applying it to a given type:

In [4]:
import cats.Show

import cats.Show


In [10]:
val showInt = Show.apply[Int]
showInt.show(4)

showInt: Show[Int] = cats.Show$$$Lambda/0x00007fca6cab7678@37ec855b
res10_1: String = "4"

In [12]:
Show[Int].show(4)

res12: String = "4"

We can use it for different types, which are of coure checked:

In [12]:
Show[Boolean].show(4)

-- [E007] Type Mismatch Error: cmd13.sc:1:31 -----------------------------------
1 |val res13 = Show[Boolean].show(4)
  |                               ^
  |      Found:    (4 : Int)
  |      Required: Boolean
  |
  |      The following import might make progress towards fixing the problem:
  |
  |        import sourcecode.Text.generate
  |
  |
  | longer explanation available when compiling with `-explain`
Compilation Failed

In [14]:
Show[Boolean].show(true)

res14: String = "true"

Cats also has lots of syntax sugar helpers that make things even simpler:

In [9]:
import cats.syntax.show.toShow

import cats.syntax.show.toShow


And now we can use `show` as an extension method:

In [17]:
4.show

res17: String = "4"

In [19]:
false.show

res19: String = "false"

In [20]:
List(4,8,7).show

res20: String = "List(4, 8, 7)"

We can also define a custom behavior for show, for example for the `Boolean` type:

In [22]:
given Show[Boolean] with
  def show(b: Boolean): String = b match
    case true => "vrai"
    case false => "faux"

defined object 

In [23]:
false.show

res23: String = "faux"

A simpler syntax is also possible:

In [25]:
given Show[Double] = Show.show(d => s"$d*10^0")

given_Show_Double: Show[Double] = <given>

In [27]:
39.45.show

res27: String = "39.45*10^0"

In [ ]:
39.4f.show

-- [E008] Not Found Error: cmd1.sc:1:17 ----------------------------------------
1 |val res1 = 39.4f.show
  |           ^^^^^^^^^^
  |value show is not a member of Float, but could be made available as an extension method.
  |
  |The following import might make progress towards fixing the problem:
  |
  |  import sourcecode.Text.generate
  |
Compilation Failed

Surely we can do it for other types as well:

In [5]:
case class Student(name:String, age:Int)

defined class Student

In [8]:
given Show[Student] = 
  Show.show(s => s"${s.name}, ${s.age} years old")

given_Show_Student: Show[Student] = <given>

In [10]:
val will = Student("will",21)
will.show

will: Student = Student(name = "will", age = 21)
res10_1: String = "will, 21 years old"

Equal

Equality with a bit of type checking

In [1]:
(4:Any) == true

res1: Boolean = false

In [3]:
case class Doctor(name:String)
case class Nurse(name:String)

defined class Doctor
defined class Nurse

In [4]:
val doris = Doctor("doris")
val doris2 = Doctor("Doris")
val nick = Nurse("nick")

doris: Doctor = Doctor(name = "doris")
doris2: Doctor = Doctor(name = "Doris")
nick: Nurse = Nurse(name = "nick")

In [5]:
doris == nick

res5: Boolean = false

In [6]:
val list:List[Any] = List(8,21,6).map(Option(_))

list: List[Any] = List(Some(value = 8), Some(value = 21), Some(value = 6))

In [7]:
list filter (i => i == 6)

res7: List[Any] = List()

In [8]:
val list2 = List(doris,nick).map(Option(_))

list2: List[Option[scala.collection.immutable.List[scala.Option[ammonite.$sess.cmd4.wrapper.cmd3.Doctor | ammonite.$sess.cmd4.wrapper.cmd3.Nurse]]]] = List(
  Some(value = Doctor(name = "doris")),
  Some(value = Nurse(name = "nick"))
)

In [9]:
list2.filter(p => p == nick)

res9: List[Option[scala.collection.immutable.List[scala.Option[ammonite.$sess.cmd4.wrapper.cmd3.Doctor | ammonite.$sess.cmd4.wrapper.cmd3.Nurse]]]] = List()

In [11]:
import cats.Eq
Eq[Int].eqv(123,123)

import cats.Eq

res11_1: Boolean = true

In [11]:
Eq[Int].eqv(123,"123")

-- [E007] Type Mismatch Error: cmd12.sc:1:28 -----------------------------------
1 |val res12 = Eq[Int].eqv(123,"123")
  |                            ^^^^^
  |Found:    ("123" : String)
  |Required: Int
  |
  |One of the following imports might make progress towards fixing the problem:
  |
  |  import os.Path.stringPathValidated
  |  import os.PathChunk.stringPathChunkValidated
  |  import os.RelPath.stringRelPathValidated
  |  import os.SubPath.stringSubPathValidated
  |  import sourcecode.Text.generate
  |
  |
  | longer explanation available when compiling with `-explain`
Compilation Failed

In [12]:
import cats.syntax.eq._

123 === 123

import cats.syntax.eq._


res12_1: Boolean = true

In [13]:
123 =!= 234

res13: Boolean = true

Custom equality

In [14]:
given Eq[Doctor] = 
  Eq.instance[Doctor]{(a,b)=>a.name.toLowerCase === b.name.toLowerCase}

given_Eq_Doctor: Eq[Doctor] = <given>

In [15]:
doris === doris2

res15: Boolean = true

In [15]:
doris === nick

-- [E007] Type Mismatch Error: cmd16.sc:1:22 -----------------------------------
1 |val res16 = doris === nick
  |                      ^^^^
  |Found:    (cmd16.this.cmd4.nick : ammonite.$sess.cmd4².wrapper.cmd3.Nurse)
  |Required: ammonite.$sess.cmd4².wrapper.cmd3.Doctor
  |
  |where:    cmd4  is a value in class cmd16
  |          cmd4² is a object in package ammonite.$sess
  |
  |
  |The following import might make progress towards fixing the problem:
  |
  |  import sourcecode.Text.generate
  |
  |
  | longer explanation available when compiling with `-explain`
Compilation Failed

Without Cats there are some type safety equality utils

In [19]:
import scala.language.strictEquality

doris == nick

import scala.language.strictEquality


res19_1: Boolean = false

In [20]:
given CanEqual[Doctor,Nurse] = CanEqual.derived

given_CanEqual_Doctor_Nurse: CanEqual[Doctor, Nurse] = <given>

In [22]:
doris == nick

res22: Boolean = false

In [22]:
nick == doris

-- [E172] Type Error: cmd23.sc:1:12 --------------------------------------------
1 |val res23 = nick == doris
  |            ^^^^^^^^^^^^^
  |Values of types ammonite.$sess.cmd4.wrapper.cmd3.Nurse and ammonite.$sess.cmd4.wrapper.cmd3.Doctor cannot be compared with == or !=
Compilation Failed

In [24]:
given CanEqual[Nurse,Doctor] = CanEqual.derived

given_CanEqual_Nurse_Doctor: CanEqual[Nurse, Doctor] = <given>

In [25]:
nick == doris

res25: Boolean = false

## Remmember Semigroup

 - non-empty set with an associative binary operation 

 - for all `x`, `y`, and `z` in the semigroup, and the binary operation `*`:

```
x * (y * z) = (x * y) * z
```

signature

```
trait Semigroup[A] :
  def combine(x: A, y: A): A 
```

In [27]:
import cats.Semigroup

Semigroup[Int].combine(34,23)

import cats.Semigroup


res27_1: Int = 57